In [ ]:
%tensorflow_version 1.x
!pip install --upgrade h5py==2.10.0
!git clone https://github.com/meghanamallarapu/Mask_RCNN
import sys
sys.path.append("/content/Mask_RCNN/demo")
from train_mask_rcnn_demo import *
%matplotlib inline

TensorFlow 1.x selected.
     |████████████████████████████████| 2.9 MB 5.3 MB/s 
  Attempting uninstall: h5py
    Found existing installation: h5py 3.1.0
    Uninstalling h5py-3.1.0:
      Successfully uninstalled h5py-3.1.0
Cloning into 'Mask_RCNN'...
remote: Enumerating objects: 245, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (240/240), done.
remote: Total 245 (delta 73), reused 89 (delta 2), pack-reused 0
Receiving objects: 100% (245/245), 122.41 MiB | 34.67 MiB/s, done.
Resolving deltas: 100% (73/73), done.
VERS 0.2


Using TensorFlow backend.


... done downloading pretrained model!


In [ ]:
images_path="/content/Mask_RCNN/dataset.zip"
annotations_path="/content/Mask_RCNN/annotations.json"
extract_images(os.path.join("/content/",images_path),"/content/dataset")

In [ ]:
dataset_train=load_image_dataset(os.path.join("/content/",annotations_path),"/content/dataset","train")
dataset_val=load_image_dataset(os.path.join("/content/",annotations_path),"/content/dataset","val")
class_number=dataset_train.count_classes()
print('Train: %d' % len(dataset_train.image_ids))
print('Validation: %d' % len(dataset_val.image_ids))
print("Classes: {}".format(class_number))

In [ ]:
config=CustomConfig(class_number)
model=load_training_model(config)

In [ ]:
train_head(model,dataset_train,dataset_train,config)


Starting at epoch 0. LR=0.002

Checkpoint Path: /content/Mask_RCNN/logs/object20220123T0929/mask_rcnn_object_{epoch:04d}.h5
Selecting layers to train
fpn_c5p5               (Conv2D)
fpn_c4p4               (Conv2D)
fpn_c3p3               (Conv2D)
fpn_c2p2               (Conv2D)
fpn_p5                 (Conv2D)
fpn_p2                 (Conv2D)
fpn_p3                 (Conv2D)
fpn_p4                 (Conv2D)
In model:  rpn_model
    rpn_conv_shared        (Conv2D)
    rpn_class_raw          (Conv2D)
    rpn_bbox_pred          (Conv2D)
mrcnn_mask_conv1       (TimeDistributed)
mrcnn_mask_bn1         (TimeDistributed)
mrcnn_mask_conv2       (TimeDistributed)
mrcnn_mask_bn2         (TimeDistributed)
mrcnn_class_conv1      (TimeDistributed)
mrcnn_class_bn1        (TimeDistributed)
mrcnn_mask_conv3       (TimeDistributed)
mrcnn_mask_bn3         (TimeDistributed)
mrcnn_class_conv2      (TimeDistributed)
mrcnn_class_bn2        (TimeDistributed)
mrcnn_mask_conv4       (TimeDistributed)
mrcnn_mask_bn

In [ ]:
test_model,inference_config=load_inference_model(1,"/content/Mask_RCNN/logs/object20220123T0929/mask_rcnn_object_0020.h5")

In [ ]:
TrDict = {'csrt': cv2.TrackerCSRT_create,
         'kcf' : cv2.TrackerKCF_create,
         'boosting' : cv2.TrackerBoosting_create,
         'mil': cv2.TrackerMIL_create,
         'tld': cv2.TrackerTLD_create,
         'medianflow': cv2.TrackerMedianFlow_create,
         'mosse':cv2.TrackerMOSSE_create}

v = cv2.VideoCapture(r'/content/drive/MyDrive/1.mp4')

fourcc = cv2.VideoWriter_fourcc(*"XVID")
output = cv2.VideoWriter("/content/output2.avi",fourcc,20.0,(640,480))
frameNumber = 1

polyp_count=0
while True:

    timer=cv2.getTickCount()

    while(polyp_count==0):

      ret, frame = v.read()
      if not ret:
        break
      r = test_model.detect([frame],verbose=1)[0]
      object_count = len(r["class_ids"])
      if(object_count!=0):

        polyp_count=1

      else:
        fps=cv2.getTickFrequency()/(cv2.getTickCount()-timer)

        cv2.putText(frame,str(int(fps)),(50,50),cv2.FONT_HERSHEY_COMPLEX,0.7,(0,0,255),2)
        output.write(frame)

    ret, frame = v.read()
    if not ret:
        break

    frame1=frame
    if ((frameNumber-1)%50) == 0:
        r = test_model.detect([frame],verbose=1)[0]
        object_count = len(r["class_ids"])
        if(object_count==0):
          fps=cv2.getTickFrequency()/(cv2.getTickCount()-timer)
          cv2.putText(frame,str(int(fps)),(50,50),cv2.FONT_HERSHEY_COMPLEX,0.7,(0,0,255),2)
          output.write(frame)
          continue

        colors = random_colors(object_count)

        for i in range(object_count):
            mask = r["masks"][:, :, i]
            contours = get_mask_contours(mask)
            for cnt in contours:
                cv2.polylines(frame, [cnt], True, colors[i], 2)
                frame = draw_mask(frame, [cnt], colors[i])
        trackers = cv2.MultiTracker_create()
        for i in range(object_count):
            bbi =(r['rois'][i][1],r['rois'][i][0],r['rois'][i][2]-r['rois'][i][0],r['rois'][i][3]-r['rois'][i][1])
            tracker_i = TrDict['kcf']()
            trackers.add(tracker_i,frame,bbi)
    (success,boxes) = trackers.update(frame)
    frameNumber+=1
    if success:
        for box in boxes:
            (x,y,w,h) = [int(a) for a in box]
            cv2.rectangle(frame,(x,y),(x+w,y+h),(100,255,0),2)
        fps=cv2.getTickFrequency()/(cv2.getTickCount()-timer)
        cv2.putText(frame,str(int(fps)),(50,50),cv2.FONT_HERSHEY_COMPLEX,0.7,(0,0,255),2)
        output.write(frame)
    else:
        fps=cv2.getTickFrequency()/(cv2.getTickCount()-timer)
        cv2.putText(frame,str(int(fps)),(50,50),cv2.FONT_HERSHEY_COMPLEX,0.7,(0,0,255),2)
        output.write(frame)
    key = cv2.waitKey(5) & 0xFF
    if key == ord('q'):
        break
v.release()
output.release()
cv2.destroyAllWindows()


In [ ]:
TrDict = {'csrt': cv2.TrackerCSRT_create,
         'kcf' : cv2.TrackerKCF_create,
         'boosting' : cv2.TrackerBoosting_create,
         'mil': cv2.TrackerMIL_create,
         'tld': cv2.TrackerTLD_create,
         'medianflow': cv2.TrackerMedianFlow_create,
         'mosse':cv2.TrackerMOSSE_create}

v = cv2.VideoCapture(r'/content/drive/MyDrive/final project/1.mp4')

fourcc = cv2.VideoWriter_fourcc(*"XVID")
output = cv2.VideoWriter("/content/output2.avi",fourcc,20.0,(640,480))
frameNumber = 0.0;s_fps=0.0;
while True:
  ret,frame=v.read();
  timer=cv2.getTickCount();
  if not ret:
    break
  r=test_model.detect([frame],verbose=1)[0]
  object_count=len(r["class_ids"])
  if(object_count==0):
    fps=cv2.getTickFrequency()/(cv2.getTickCount()-timer);s_fps+=fps;frameNumber+=1;
    output.write(frame);
    for i in range(0,3):
      ret,frame=v.read();
      #timer=cv2.getTickCount();
      if not ret:
        break
      fps=cv2.getTickFrequency()/(cv2.getTickCount()-timer);s_fps+=fps;frameNumber+=1;
      output.write(frame);
  else:
    colors=visualize.random_colors(object_count)
    for i in range(object_count):
      mask=r["masks"][:,:,i]
      contours=visualize.get_mask_contours(mask)
      for cnt in contours:
        cv2.polylines(frame,[cnt],True,colors[i],2)
        frame=visualize.draw_mask(frame,[cnt],colors[i])
    fps=cv2.getTickFrequency()/(cv2.getTickCount()-timer);s_fps+=fps;frameNumber+=1;
    output.write(frame);
    trackers=cv2.MultiTracker_create()
    for i in range(object_count):
        bbi=(r['rois'][i][1],r['rois'][i][0],r['rois'][i][2]-r['rois'][i][0],r['rois'][i][3]-r['rois'][i][1])
        tracker_i=TrDict['csrt']()
        trackers.add(tracker_i,frame,bbi)
    for j in range(0,9):
      (success,boxes)=trackers.update(frame);
      if success:
        for box in boxes:
          (x,y,w,h) = [int(a) for a in box]
          cv2.rectangle(frame,(x,y),(x+w,y+h),(100,255,0),2)
        fps=cv2.getTickFrequency()/(cv2.getTickCount()-timer);s_fps+=fps;frameNumber+=1;
        output.write(frame);
      else:
        break;
      ret,frame=v.read();
      #timer=cv2.getTickCount();
      if not ret:
        break;
time=cv2.getTickCount()-timer;
v.release();output.release();cv2.destroyAllWindows
fps=s_fps/frameNumber;
print(fps);

Processing 1 images
image                    shape: (480, 640, 3)         min:    0.00000  max:  255.00000  uint8
molded_images            shape: (1, 512, 512, 3)      min: -123.70000  max:  151.10000  float64
image_metas              shape: (1, 14)               min:    0.00000  max:  640.00000  float64
anchors                  shape: (1, 65472, 4)         min:   -0.70849  max:    1.58325  float32
Processing 1 images
image                    shape: (480, 640, 3)         min:    0.00000  max:  255.00000  uint8
molded_images            shape: (1, 512, 512, 3)      min: -123.70000  max:  151.10000  float64
image_metas              shape: (1, 14)               min:    0.00000  max:  640.00000  float64
anchors                  shape: (1, 65472, 4)         min:   -0.70849  max:    1.58325  float32
Processing 1 images
image                    shape: (480, 640, 3)         min:    0.00000  max:  255.00000  uint8
molded_images            shape: (1, 512, 512, 3)      min: -123.70000  max:  151.1

In [ ]:
import cv2
cap=cv2.VideoCapture('/content/non polyp2.mp4');i=0;
while(cap.isOpened()):
  ret,frame=cap.read()
  if ret==False:
    break
  cv2.imwrite('/content/fold/f'+str(i)+'.jpg',frame)
  i+=1;
cap.release();cv2.destroyAllWindows()

In [ ]:
import os
import zipfile
folder_path='/content/fold';zip_path="/content/Folder.zip"
with zipfile.ZipFile(zip_path,mode='w') as zipf:
  len_dir_path=len(folder_path)
  for root, _,files in os.walk(folder_path):
    for file in files:
      file_path=os.path.join(root,file)
      zipf.write(file_path,file_path[len_dir_path:])